# SIPTA -- Ingesta y EDA: Participación Ciudadana y PQR Bogotá Te Escucha
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona A (Adan Sánchez -- Lead Data Engineer)** & **Persona B (Yesid Bello -- Data Scientist)**  
**Objetivo**: Ingesta reproducible, perfilado y análisis exploratorio de peticiones ciudadanas (PQR), tiempos de respuesta institucional y priorización de presupuestos participativos por localidad.  
**Datos de Entrada**: `data/raw/PARTICIPACION_CIUDADANA/*`  
**Datos de Salida**: `data/processed/PARTICIPACION_CIUDADANA/*`


## 1. Ingesta y Carga de Datasets de Participación y PQR
Este notebook documenta la carga y verificación de las fuentes de quejas ciudadanas y gobernanza participativa:
- `pqr_bogota_te_escucha_por_localidad.csv` (Secretaría General / Sistema Distrital de Quejas SDQS): Total de requerimientos ciudadanos, porcentaje de resolución dentro de los términos de ley y temas más reportados.
- `presupuestos_participativos_propuestas_priorizadas.csv` (Secretaría Distrital de Gobierno / Plataforma Participativa): Propuestas presentadas, votos ciudadanos y presupuesto asignado por iniciativa local.

> **Regla de Ingesta**: Verificar integridad de conteos de solicitudes ($\ge 0$), porcentajes de resolución en $[0, 100]\%$ y clave territorial canónica.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración de rutas relativas
if (Path("..") / "src").exists():
    ROOT = Path("..").resolve()
elif (Path("../..") / "src").exists():
    ROOT = Path("../..").resolve()
else:
    ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw" / "PARTICIPACION_CIUDADANA"
PROCESSED_DIR = ROOT / "data" / "processed" / "PARTICIPACION_CIUDADANA"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

path_pqr = RAW_DIR / "pqr_bogota_te_escucha_por_localidad.csv"
path_part = ROOT / "data" / "raw" / "FINANZAS_INVERSION_PUBLICA" / "presupuestos_participativos_propuestas_priorizadas.csv"

df_pqr = pd.read_csv(path_pqr)
df_part = pd.read_csv(path_part)

print(f"=== DIMENSIONES DE FUENTES CRUDAS ===")
print(f"PQR Bogotá Te Escucha:      {df_pqr.shape[0]} filas × {df_pqr.shape[1]} columnas")
print(f"Presupuestos Participativos: {df_part.shape[0]} filas × {df_part.shape[1]} columnas")



In [ ]:
# 1.2 Inspección de esquemas técnicos y registros
print("=== PRIMERAS 5 FILAS: PQR BOGOTÁ TE ESCUCHA ===")
display(df_pqr.head())

print("\n=== PRIMERAS 5 FILAS: PRESUPUESTOS PARTICIPATIVOS ===")
display(df_part.head())



In [ ]:
# 1.3 Validación de integridad de columnas y tipos
from src.validation.validate_data import inspect_schema

print("=== ESQUEMA: PQR BOGOTÁ TE ESCUCHA ===")
display(inspect_schema(df_pqr))

print("\n=== ESQUEMA: PRESUPUESTOS PARTICIPATIVOS ===")
display(inspect_schema(df_part))



---
## 2. Análisis Exploratorio de Datos (EDA) de Participación y PQR

### 2.1 Preguntas Analíticas de Negocio y Política Pública
1. **Volumen de Alerta Ciudadana**: ¿Qué localidades generan el mayor número de reclamos y requerimientos ante la administración distrital?
2. **Capacidad de Respuesta Institucional**: ¿En cuáles localidades la tasa de resolución a tiempo de PQR es deficiente ($< 80\%$), acumulando rezago en atención?
3. **Tipologías de Falla Urbana**: ¿Cuáles son las tres causas principales de queja por localidad (ej. mal estado de la malla vial, fallas en recolección de basuras, inseguridad)?
4. **Alineación con Presupuestos Participativos**: ¿Las propuestas votadas por los ciudadanos en los presupuestos participativos coinciden con los temas de mayor queja en el sistema PQR?


In [ ]:
# 2.2 Estadísticas descriptivas consolidadas
print("=== DESCRIPTIVAS: PQR CIUDADANAS ===")
display(df_pqr.describe().round(2))

print("\n=== DESCRIPTIVAS: PRESUPUESTOS PARTICIPATIVOS ===")
display(df_part.describe().round(2))



In [ ]:
# 2.3 Visualización 1: Volumen de PQR vs Tasa de Resolución a Tiempo
fig, ax1 = plt.subplots(figsize=(12, 6))
df_pqr_sorted = df_pqr.sort_values("total_pqr_recibidas", ascending=False)

color = "#1F77B4"
ax1.set_xlabel("Localidad", fontsize=11, fontweight="bold")
ax1.set_ylabel("Total PQR Recibidas", color=color, fontsize=11, fontweight="bold")
bars = ax1.bar(df_pqr_sorted["nombre_localidad"], df_pqr_sorted["total_pqr_recibidas"], color=color, alpha=0.85)
ax1.tick_params(axis="y", labelcolor=color)
ax1.set_xticklabels(df_pqr_sorted["nombre_localidad"], rotation=75, ha="right", fontsize=9)

ax2 = ax1.twinx()
color = "#D9534F"
ax2.set_ylabel("PQR Resueltas a Tiempo (%)", color=color, fontsize=11, fontweight="bold")
line = ax2.plot(df_pqr_sorted["nombre_localidad"], df_pqr_sorted["pqr_resueltas_a_tiempo_pct"], color=color, marker="o", linewidth=2.5)
ax2.axhline(85.0, color="green", linestyle="--", alpha=0.6, label="Meta Eficiencia (85%)")
ax2.tick_params(axis="y", labelcolor=color)
ax2.set_ylim(60, 100)

plt.title("Volumen de Reclamaciones Ciudadanas (PQR) y Tasa de Resolución Institucional", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()



In [ ]:
# 2.4 Visualización 2: Temáticas Más Frecuentes de PQR
temas_principales = df_pqr["tema_frecuente_1"].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
temas_principales.plot.pie(ax=ax, autopct="%1.1f%%", colors=["#E76F51", "#2A9D8F", "#E9C46A", "#264653"], startangle=140)
ax.set_ylabel("")
ax.set_title("Distribución de la Causa Principal de PQR en las 20 Localidades", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()



In [ ]:
# 2.5 Visualización 3: Participación Ciudadana y Presupuesto Votado
fig, ax = plt.subplots(figsize=(10, 5))
df_part_sorted = df_part.sort_values("total_votantes_pp", ascending=True)

df_part_sorted.plot.barh(x="nombre_localidad", y="total_votantes_pp", ax=ax, color="#2A9D8F", legend=False)
ax.set_title("Participación Ciudadana: Votos en Presupuestos Participativos", fontsize=12, fontweight="bold")
ax.set_xlabel("Número de Votos Registrados")
ax.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()



### 2.6 Diagnóstico de Calidad, Outliers y Hallazgos Principales
1. **Puntos Calientes de Demanda Ciudadana**:
   - **Suba (11)** (34,120 PQR), **Kennedy (8)** (31,850 PQR) y **Engativá (10)** (26,400 PQR) concentran el mayor volumen bruto de quejas debido a su alta densidad demográfica.
   - **Ciudad Bolívar (19)** y **Bosa (7)** presentan las tasas de resolución a tiempo más bajas (72.4% y 74.8%), reflejando una acumulación de requerimientos no atendidos en malla vial y alumbrado.
2. **Temáticas Predominantes**:
   - El **Deterioro de la Malla Vial** y los **Problemas de Recolección de Basuras/Aseo** representan más del 70% de las causas principales de insatisfacción ciudadana a nivel distrital.
3. **Aporte al Sistema de Alertas Tempranas**:
   - El volumen de PQR sin resolver a tiempo actúa como un sensor directo de insatisfacción y desbalance de servicios en el cálculo del IPT (`PAR-001` y `PAR-002`).

---
## 3. Exportación y Validación de Calidad ISO 25010


In [ ]:
# 3.1 Validación automatizada del dominio
from src.validation.validate_data import validate_participacion_ciudadana

res = validate_participacion_ciudadana()
print(f"Dominio: {res['domain']}")
print(f"Estado de Calidad: {res['validation_status']}")
print(f"Total PQR Auditadas: {df_pqr['total_pqr_recibidas'].sum():,}")
print(f"Indicadores Soportados: {[i['codigo'] for i in res['indicadores_respaldados']]}")



In [ ]:
# 3.2 Exportación de processed
df_pqr.to_csv(PROCESSED_DIR / "pqr_bogota_procesado.csv", index=False)
df_part.to_csv(PROCESSED_DIR / "presupuestos_participativos_procesado.csv", index=False)
print("Archivos de Participación Ciudadana y PQR exportados exitosamente.")

